In [ ]:
from ultralytics import YOLO
import torch
import os

def train_orange_detector():
    # 1. Configuración de Semilla para Reproducibilidad (Como en el paper)
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False # Desactivar para resultados exactos, activar para velocidad

    # 2. Definir Rutas
    # Asegúrate de que 'dataset/data.yaml' apunte correctamente a tus carpetas
    data_yaml_path = 'dataset/data.yaml'
    project_name = 'orange_miniproject'
    experiment_name = 'yolov8n_adam_cos'

    # 3. Cargar Modelo Pre-entrenado (Transfer Learning)
    # Usamos 'yolov8n.pt' (nano) por ser el más rápido y ligero, ideal para tu mini-proyecto
    print("Cargando modelo YOLOv8n pre-entrenado en COCO...")
    model = YOLO('yolov8n.pt')

    # 4. Entrenamiento con Hiperparámetros Óptimos
    # Basado en Khedkar et al. (2025): Adam optimizer, lr=0.001, momentum característico
    print(f"Iniciando entrenamiento en: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

    results = model.train(
        data=data_yaml_path,

        # --- Parámetros de Entrenamiento ---
        epochs=200,                # El paper sugiere 200, es un buen estándar
        patience=25,               # Early Stopping: si no mejora en 25 épocas, para.
        batch=16,                  # Tamaño de batch estándar para GPU media (T4/3060)
        imgsz=640,                 # Resolución de entrada (640x640)

        # --- Optimizador y Learning Rate (Clave del paper) ---
        optimizer='Adam',          # Khedkar encontró que Adam funcionaba mejor que SGD aquí
        lr0=0.001,                 # Tasa de aprendizaje inicial
        lrf=0.01,                  # Tasa final (lr0 * lrf)
        momentum=0.937,            # Momentum para Adam (beta1)
        weight_decay=0.0005,       # Regularización para evitar overfitting
        cos_lr=True,               # Scheduler tipo Coseno (suaviza la bajada del LR)

        # --- Data Augmentation (Configuración interna de YOLO) ---
        # YOLOv8 hace augmentación al vuelo (on-the-fly), no necesitas generar imágenes antes.
        # Aquí replicamos la configuración "lógica" del paper y corregimos la rotación.
        degrees=15.0,              # Rotación +/- 15 grados (NO 90)
        fliplr=0.5,                # Espejo horizontal 50%
        flipud=0.0,                # Espejo vertical (Apagado para fotos terrestres, encender (0.5) si es Dron cenital)
        scale=0.5,                 # Escalar imagen (+/- 50%)
        mosaic=1.0,                # Mosaic activado (ayuda mucho a detectar objetos pequeños)
        mixup=0.1,                 # Mezclar imágenes (ayuda a la robustez)
        hsv_h=0.015,               # Variación leve de tono (Hue)
        hsv_s=0.7,                 # Variación media de saturación (Saturation)
        hsv_v=0.4,                 # Variación media de valor (Brillo)

        # --- Hardware y Guardado ---
        device=0,                  # GPU 0 (cambia a 'cpu' si no tienes GPU NVIDIA)
        workers=4,                 # Hilos para cargar datos
        project=project_name,      # Carpeta donde se guardan resultados
        name=experiment_name,      # Nombre de la subcarpeta
        exist_ok=True,             # Sobrescribir si existe
        seed=42,                   # Semilla determinista
        plots=True                 # Generar gráficas de entrenamiento automáticamente
    )

    print("Entrenamiento finalizado.")
    print(f"Mejor modelo guardado en: {project_name}/{experiment_name}/weights/best.pt")

    # 5. Validación automática
    metrics = model.val()
    print(f"mAP50: {metrics.box.map50}")
    print(f"mAP50-95: {metrics.box.map}")

if __name__ == '__main__':
    # Verificar si existe el archivo data.yaml
    if not os.path.exists('dataset/data.yaml'):
        print("ERROR: No se encuentra 'dataset/data.yaml'. Por favor crea la estructura de carpetas.")
        # Creamos un dummy yaml para que veas el formato si no existe
        with open('dataset_example.yaml', 'w') as f:
            f.write("path: ../dataset\ntrain: train/images\nval: val/images\ntest: test/images\n\nnames:\n  0: naranja")
        print("Se ha creado un ejemplo 'dataset_example.yaml'. Úsalo de guía.")
    else:
        train_orange_detector()